In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/VuTrinhNguyenHoang/bidirectional-table-text-alignment.git

In [ ]:
%cd bidirectional-table-text-alignment

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from pathlib import Path
import json

# "small", "medium", "debug", "full"
MODE = "debug"
GENERATOR_MODE = "debug" 
SELECTOR_MODE = "debug"
CONFIG = "configs/main.yaml"
TOP_K = 3
THRESHOLD_STEP = 0.01

In [ ]:
import yaml

config = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))

# model: "google-t5/t5-small", "google/flan-t5-small", "google/t5-v1_1-small"
# cell_selector: "distilbert-base-uncased", "microsoft/MiniLM-L12-H384-uncased", "google/electra-small-discriminator"

config["model"]["name"] = "google-t5/t5-small"
config["cell_selector"]["name"] = "distilbert-base-uncased"

Path(CONFIG).write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

In [ ]:
!PYTHONPATH=. python scripts/prepare_subset.py \
    --mode {MODE} \
    --generator-mode {GENERATOR_MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/tokenize_dataset.py \
    --mode {MODE} \
    --generator-mode {GENERATOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/train_generator.py \
    --mode {MODE} \
    --generator-mode {GENERATOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/evaluation_generation.py \
    --mode {MODE} \
    --generator-mode {GENERATOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/run_faithfulness.py \
    --mode {MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/prepare_cell_dataset.py \
    --mode {MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/tokenize_cell_dataset.py \
    --mode {MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/train_cell_selector.py \
    --mode {MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG}

In [ ]:
!PYTHONPATH=. python3 scripts/tune_cell_selector_threshold.py \
    --mode {MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG} \
    --top_k {TOP_K} \
    --split valid \
    --threshold_step {THRESHOLD_STEP}

In [ ]:
tuning_path = Path(f"outputs/metrics/{MODE}/cell_selector_threshold_tuning_valid_top{TOP_K}.json")
tuning = json.loads(tuning_path.read_text(encoding="utf-8"))
BEST_THRESHOLD = tuning["best_threshold"]
BEST_THRESHOLD, tuning["best_metrics"]

In [ ]:
!PYTHONPATH=. python3 scripts/evaluation_e2e.py \
    --mode {MODE} \
    --generator-mode {GENERATOR_MODE} \
    --selector-mode {SELECTOR_MODE} \
    --config {CONFIG} \
    --top_k {TOP_K} \
    --threshold {BEST_THRESHOLD}

In [ ]:
threshold_tag = f"threshold{BEST_THRESHOLD:.4f}".replace(".", "p")
metric_path = Path(f"outputs/metrics/{MODE}/e2e_metrics_{threshold_tag}_top{TOP_K}.json")
metrics = json.loads(metric_path.read_text(encoding="utf-8"))
metrics